# Codebase Function Guide

This notebook summarizes the Python files in the weather probability modeling repo, lists every class and function in those files, and demonstrates the main workflows with the current NYC Open-Meteo data.

The current modeling target remains:

```python
forecast_error = actual_high - forecast_high
```

Important caveat: the forecast data is treated as an Open-Meteo historical forecast proxy, not confirmed official NWS archived forecast data.

## Import Setup

Run this first. It makes imports work whether the notebook is opened from the repo root or from inside `notebooks/`.

In [ ]:
from __future__ import annotations

import ast
import inspect
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

repo_root

## File Inventory

This cell summarizes each Python source file. Generated cache files are intentionally excluded.

In [ ]:
SOURCE_FILES = [
    Path("src/__init__.py"),
    Path("src/bucket_schema.py"),
    Path("src/data_audit.py"),
    Path("src/distribution_pricing.py"),
    Path("src/error_boundaries.py"),
    Path("src/forecast_data.py"),
    Path("src/main_demo.py"),
    Path("src/visuals.py"),
    Path("src/weather_data.py"),
    Path("scripts/inspect_day5_data.py"),
    Path("scripts/run_day6_data_verification.py"),
    Path("tests/distribution_pricing_tests.py"),
]

FILE_SUMMARIES = {
    "src/__init__.py": "Marks src as an importable package.",
    "src/bucket_schema.py": "Defines Kalshi-style temperature buckets and validates bucket coverage.",
    "src/data_audit.py": "Loads CSV files, audits schemas/date ranges/missingness, and writes Day 6 inventory/report outputs.",
    "src/distribution_pricing.py": "Prices bucket intervals from a normal distribution and saves a CDF plot.",
    "src/error_boundaries.py": "Converts actual-temperature market buckets into forecast-error intervals.",
    "src/forecast_data.py": "Loads, standardizes, and validates Open-Meteo forecast data.",
    "src/main_demo.py": "Small command-line demo for bucket resolution and error-boundary conversion.",
    "src/visuals.py": "Visualization helpers for CDF curves and bucket-probability regions.",
    "src/weather_data.py": "Loads, standardizes, and validates actual Open-Meteo weather data.",
    "scripts/inspect_day5_data.py": "Day 5 inspection script for loading raw data and constructing forecast error.",
    "scripts/run_day6_data_verification.py": "Day 6 end-to-end audit, cleaning, alignment, and report-generation script.",
    "tests/distribution_pricing_tests.py": "Script-style assertions for bucket probabilities and demo output generation.",
}

file_rows = []
for relative_path in SOURCE_FILES:
    path = repo_root / relative_path
    source = path.read_text(encoding="utf-8") if path.exists() else ""
    tree = ast.parse(source) if source else ast.Module(body=[])
    top_level_defs = [node for node in tree.body if isinstance(node, (ast.FunctionDef, ast.ClassDef))]
    file_rows.append(
        {
            "file": str(relative_path),
            "lines": source.count("\n") + 1 if source else 0,
            "functions/classes": len(top_level_defs),
            "summary": FILE_SUMMARIES.get(str(relative_path), "No summary provided."),
        }
    )

pd.DataFrame(file_rows)

## Function And Class Inventory

The next cell lists every top-level function and class in the source files. Public functions are the intended API; private helpers are included because they explain how the scripts are assembled.

In [ ]:
FUNCTION_SUMMARIES = {
    "Bucket": "Dataclass representing one temperature bucket with open or finite bounds.",
    "validate_buckets": "Checks bucket order, gaps, overlaps, and open-ended first/last buckets.",
    "bucket_for_actual_temp": "Finds the one bucket that contains a realized actual high.",
    "format_bucket_interval": "Formats a bucket as a human-readable actual-temperature interval.",
    "load_csv": "Reads regular CSVs and Open-Meteo CSVs with metadata preambles.",
    "read_openmeteo_metadata": "Extracts coordinate/timezone metadata from Open-Meteo CSV preamble rows.",
    "infer_datetime_column": "Chooses the most likely date/time column from a dataframe.",
    "summarize_file": "Builds one audit row with shape, dates, duplicates, columns, and missingness.",
    "create_data_inventory": "Audits a list of CSVs and writes outputs/data_inventory.csv.",
    "write_verification_report": "Writes the Day 6 markdown verification report.",
    "normal_cdf": "Evaluates a normal CDF for scalar or array inputs.",
    "plot_and_save_cdf": "Saves a basic normal CDF figure under outputs/figures.",
    "normal_bucket_prob": "Computes probability mass between lower and upper bounds.",
    "normal_bucket_probs": "Computes probabilities for a full list of buckets.",
    "ErrorInterval": "Dataclass representing one forecast-error interval for one market bucket.",
    "bucket_to_error_interval": "Converts one actual-temperature bucket into an error interval.",
    "convert_market_to_boundaries": "Builds Kalshi-style buckets from sorted market labels.",
    "convert_nws_to_boundaries": "Builds a six-bucket market centered around one forecast high.",
    "buckets_to_error_intervals": "Converts every bucket in a market into forecast-error intervals.",
    "extract_cdf_boundaries": "Extracts finite error cutpoints needed for CDF pricing.",
    "format_error_interval": "Formats a forecast-error interval for display.",
    "identify_forecast_high_column": "Finds the forecast high column in raw or cleaned forecast data.",
    "load_hourly_forecasts": "Loads hourly Open-Meteo forecast CSVs in the older Day 5 shape.",
    "load_daily_forecasts": "Loads daily Open-Meteo forecast CSVs in the older Day 5 shape.",
    "standardize_daily_forecasts": "Creates date/location/forecast_high/forecast_source cleaned daily forecast rows.",
    "standardize_hourly_forecasts": "Creates timestamp/date/location cleaned hourly forecast rows.",
    "validate_forecast_values": "Returns warnings for missing keys, duplicates, impossible values, and as-of gaps.",
    "validate_hourly_forecasts": "Raises on old-style hourly forecast structural problems.",
    "validate_daily_forecasts": "Raises on old-style daily forecast structural problems.",
    "format_bound": "Formats signed numeric bounds for the command-line demo.",
    "build_demo_buckets": "Builds a fixed demo market for a location.",
    "run_manual_resolution_checks": "Prints sample actual highs and their resolved buckets.",
    "main": "Command-line entry point for the file it appears in.",
    "get_bounds": "Returns sorted unique finite bucket boundaries.",
    "plot_cdf_with_probabilities": "Saves a CDF figure shaded by bucket probabilities.",
    "standardize_openmeteo_columns": "Renames Open-Meteo columns while tolerating unit suffix differences.",
    "identify_actual_high_column": "Finds the actual daily high column in raw or cleaned weather data.",
    "load_hourly_weather": "Loads hourly actual Open-Meteo weather in the older Day 5 shape.",
    "load_daily_weather": "Loads daily actual Open-Meteo weather in the older Day 5 shape.",
    "standardize_daily_weather": "Creates date/location/actual_high cleaned daily actual-weather rows.",
    "standardize_hourly_weather": "Creates timestamp/date/location cleaned hourly actual-weather rows.",
    "validate_weather_values": "Returns warnings for missing keys, duplicates, impossible values, and missing sections.",
    "validate_hourly_weather": "Raises on old-style hourly actual-weather structural problems.",
    "validate_daily_weather": "Raises on old-style daily actual-weather structural problems.",
    "load_module_from_path": "Loads a Python module directly from a path for script-style tests.",
    "generate_bucket_probability_demo": "Writes a small bucket-probability demo CSV used by the test script.",
}

def signature_for_node(node: ast.AST) -> str:
    if isinstance(node, ast.ClassDef):
        return f"class {node.name}"
    if isinstance(node, ast.FunctionDef):
        args = [arg.arg for arg in node.args.args]
        if node.args.vararg:
            args.append("*" + node.args.vararg.arg)
        if node.args.kwarg:
            args.append("**" + node.args.kwarg.arg)
        return f"{node.name}({', '.join(args)})"
    return ""

def fallback_summary(name: str) -> str:
    if name.startswith("_append_"):
        return "Private helper that appends validation/report warnings."
    if name.startswith("_format_"):
        return "Private helper for report/display formatting."
    if name.startswith("_read_"):
        return "Private CSV-reading helper."
    if name.startswith("_validate_"):
        return "Private validation helper."
    if name.startswith("_"):
        return "Private helper used by this module's public workflow."
    return "See signature and source file for details."

function_rows = []
for relative_path in SOURCE_FILES:
    path = repo_root / relative_path
    if not path.exists():
        continue
    tree = ast.parse(path.read_text(encoding="utf-8"))
    for node in tree.body:
        if not isinstance(node, (ast.FunctionDef, ast.ClassDef)):
            continue
        docstring = ast.get_docstring(node) or ""
        summary = FUNCTION_SUMMARIES.get(node.name) or (docstring.split("\n")[0] if docstring else fallback_summary(node.name))
        function_rows.append(
            {
                "file": str(relative_path),
                "line": node.lineno,
                "kind": "class" if isinstance(node, ast.ClassDef) else "function",
                "name": node.name,
                "visibility": "private" if node.name.startswith("_") else "public",
                "signature": signature_for_node(node),
                "summary": summary,
            }
        )

function_inventory = pd.DataFrame(function_rows)
function_inventory

## Demo 1: Audit Raw CSV Files

This demonstrates `src.data_audit`: loading Open-Meteo CSVs with metadata rows, inferring date columns, summarizing missingness, and writing a demo inventory file.

In [ ]:
from src.data_audit import (
    create_data_inventory,
    infer_datetime_column,
    load_csv,
    read_openmeteo_metadata,
    summarize_file,
)

raw_paths = {
    "daily_actual": repo_root / "data/raw/daily_raw_nyc_openmeteo.csv",
    "hourly_actual": repo_root / "data/raw/hourly_raw_nyc_openmeteo.csv",
    "daily_forecast": repo_root / "data/forecasts/daily_forecasts_nyc_openmeteo.csv",
    "hourly_forecast": repo_root / "data/forecasts/hourly_forecasts_nyc_openmeteo.csv",
}

daily_actual_raw = load_csv(raw_paths["daily_actual"])
daily_forecast_raw = load_csv(raw_paths["daily_forecast"])

audit_demo = pd.DataFrame(
    [
        summarize_file(raw_paths["daily_actual"], daily_actual_raw),
        summarize_file(raw_paths["daily_forecast"], daily_forecast_raw),
    ]
)[["file", "rows", "columns", "date_time_column", "date_min", "date_max", "missing_values_total"]]

demo_inventory_path = repo_root / "outputs/notebook_data_inventory_demo.csv"
create_data_inventory(list(raw_paths.values()), demo_inventory_path)

print("Inferred daily actual datetime column:", infer_datetime_column(daily_actual_raw))
print("Daily actual metadata:", read_openmeteo_metadata(raw_paths["daily_actual"]))
print("Demo inventory written to:", demo_inventory_path)
audit_demo

## Demo 2: Clean Weather And Forecast Data

This demonstrates `src.weather_data` and `src.forecast_data`: identify high-temperature columns, standardize raw data, validate values, merge actuals with forecasts, and compute `forecast_error`.

In [ ]:
from src.forecast_data import (
    identify_forecast_high_column,
    standardize_daily_forecasts,
    standardize_hourly_forecasts,
    validate_forecast_values,
)
from src.weather_data import (
    identify_actual_high_column,
    standardize_daily_weather,
    standardize_hourly_weather,
    validate_weather_values,
)

hourly_actual_raw = load_csv(raw_paths["hourly_actual"])
hourly_forecast_raw = load_csv(raw_paths["hourly_forecast"])

daily_clean = standardize_daily_weather(daily_actual_raw, location="NYC")
hourly_clean = standardize_hourly_weather(hourly_actual_raw, location="NYC")
forecasts_clean = standardize_daily_forecasts(daily_forecast_raw, location="NYC")
hourly_forecasts_clean = standardize_hourly_forecasts(hourly_forecast_raw, location="NYC")

validation_warnings = []
validation_warnings.extend(validate_weather_values(daily_clean, "daily"))
validation_warnings.extend(validate_weather_values(hourly_clean, "hourly"))
validation_warnings.extend(validate_forecast_values(forecasts_clean, "daily"))
validation_warnings.extend(validate_forecast_values(hourly_forecasts_clean, "hourly"))

modeling_preview = daily_clean.merge(
    forecasts_clean,
    on=["date", "location"],
    how="inner",
    suffixes=("_actual", "_forecast"),
)
modeling_preview["forecast_error"] = modeling_preview["actual_high"] - modeling_preview["forecast_high"]

print("Actual high column:", identify_actual_high_column(daily_actual_raw))
print("Forecast high column:", identify_forecast_high_column(daily_forecast_raw))
print("Daily clean rows:", len(daily_clean))
print("Forecast clean rows:", len(forecasts_clean))
print("Merged rows:", len(modeling_preview))
print("Validation warnings:")
for warning in validation_warnings:
    print("-", warning)

modeling_preview[["date", "location", "actual_high", "forecast_high", "forecast_error", "forecast_source"]].head()

## Demo 3: Bucket Schema And Error Boundaries

This demonstrates `src.bucket_schema`, `src.error_boundaries`, and `src.main_demo`: build a market, validate it, resolve actual temperatures into buckets, and convert buckets into error intervals.

In [ ]:
from src.bucket_schema import bucket_for_actual_temp, format_bucket_interval, validate_buckets
from src.error_boundaries import (
    buckets_to_error_intervals,
    convert_market_to_boundaries,
    convert_nws_to_boundaries,
    extract_cdf_boundaries,
    format_error_interval as format_error_interval_from_module,
)
from src.main_demo import build_demo_buckets, format_bound

market_buckets = convert_market_to_boundaries([69, 70, 71, 72, 73, 74, 75, 76, 77, 78], "NYC")
validate_buckets(market_buckets)

forecast_high = 73.0
error_intervals = buckets_to_error_intervals(market_buckets, forecast_high)

bucket_demo = pd.DataFrame(
    {
        "bucket": [bucket.name for bucket in market_buckets],
        "actual_interval": [format_bucket_interval(bucket) for bucket in market_buckets],
        "error_interval": [format_error_interval_from_module(interval) for interval in error_intervals],
    }
)

print("Example actual_high=72.4 resolves to:", bucket_for_actual_temp(72.4, market_buckets).name)
print("CDF boundaries:", extract_cdf_boundaries(error_intervals))
print("NWS-centered demo buckets:", [bucket.name for bucket in convert_nws_to_boundaries(73, "NYC")])
print("main_demo.format_bound(2.5):", format_bound(2.5))
print("main_demo.build_demo_buckets('NYC') count:", len(build_demo_buckets("NYC")))
bucket_demo

## Demo 4: Distribution Pricing And Visuals

This demonstrates `src.distribution_pricing` and `src.visuals`: compute normal probabilities for bucket intervals and save CDF figures.

In [ ]:
from src.distribution_pricing import normal_bucket_prob, normal_bucket_probs, normal_cdf, plot_and_save_cdf
from src.visuals import get_bounds, plot_cdf_with_probabilities

mu = 73.0
sigma = 2.0
probabilities = normal_bucket_probs(market_buckets, mu=mu, sigma=sigma)

plot_and_save_cdf(mu=mu, sigma=sigma, filename="notebook_normal_cdf.png")
plot_cdf_with_probabilities(market_buckets, mu=mu, sigma=sigma, filename="notebook_cdf_shaded_probs.png")

pricing_demo = pd.DataFrame(
    {
        "bucket": list(probabilities.keys()),
        "probability": list(probabilities.values()),
    }
)

print("normal_cdf(73, mu=73, sigma=2):", normal_cdf(73, mu=mu, sigma=sigma))
print("P(error/temperature region <= first bound):", normal_bucket_prob(None, get_bounds(market_buckets)[0], mu, sigma))
print("Finite bounds:", get_bounds(market_buckets))
pricing_demo

## Demo 5: Day 6 Script Helpers

This demonstrates the pure helper functions in `scripts/run_day6_data_verification.py`. The final cell shows how to run the full script from the notebook.

In [ ]:
from scripts.run_day6_data_verification import (
    _columns_matching,
    _date_range,
    _describe_series,
    _looks_celsius_like,
    _timestamp_range,
    _unit_from_column,
)

script_helper_demo = {
    "daily_date_range": _date_range(daily_clean, "date"),
    "hourly_timestamp_range": _timestamp_range(hourly_clean, "timestamp"),
    "forecast_error_summary": _describe_series(modeling_preview["forecast_error"]),
    "unit_from_column": _unit_from_column("temperature_2m_max (F)"),
    "matching_columns": _columns_matching(list(daily_actual_raw.columns), ["temperature", "wind"]),
    "actual_high_celsius_like": _looks_celsius_like(daily_clean["actual_high"]),
}

script_helper_demo

In [ ]:
# Optional end-to-end run from the notebook. This rewrites the Day 6 outputs.
run_full_day6_script = False

if run_full_day6_script:
    completed = subprocess.run(
        [sys.executable, "scripts/run_day6_data_verification.py"],
        cwd=repo_root,
        text=True,
        capture_output=True,
        check=True,
    )
    print(completed.stdout)
else:
    print("Set run_full_day6_script=True to run the full Day 6 verification script.")

## Demo 6: Day 5 Script And Test Script Entry Points

These cells show the existing script-style entry points without changing raw data. They are kept separate because they print reports and may write demo output files.

In [ ]:
from tests.distribution_pricing_tests import generate_bucket_probability_demo

demo_probability_path = repo_root / "outputs/notebook_bucket_probability_demo.csv"
demo_rows = generate_bucket_probability_demo(demo_probability_path)

print("Rows written:", len(demo_rows))
print("Output:", demo_probability_path)
pd.DataFrame(demo_rows)

In [ ]:
# Optional Day 5 inspection run. It prints a longer report.
run_day5_inspection = False

if run_day5_inspection:
    completed = subprocess.run(
        [sys.executable, "scripts/inspect_day5_data.py"],
        cwd=repo_root,
        text=True,
        capture_output=True,
        check=True,
    )
    print(completed.stdout)
else:
    print("Set run_day5_inspection=True to run scripts/inspect_day5_data.py.")

## Where To Go Next

- Use `data/processed/modeling_base_preview.csv` as the traceable starting point for future modeling rows.
- Keep future feature engineering point-in-time safe. Do not use actual weather variables or target-day realized values as pre-event features.
- Replace the Open-Meteo historical forecast proxy with true archived NWS forecasts before making claims about NWS forecast-error performance.